# Training Notebook — Kleding Classifier

Dit notebook traint het bestaande kledingmodel bij op jouw eigen foto's (transfer learning).
Het model leert zo jouw specifieke foto-stijl herkennen.

## Mappenstructuur

Maak deze structuur aan naast dit notebook **voordat** je begint:

```
kleding_training/
    t-shirt/         <- minimaal 20 foto's
    broek/           <- minimaal 20 foto's
    jas/             <- minimaal 20 foto's
    jurk/            <- minimaal 20 foto's (optioneel)
    schoenen/        <- minimaal 20 foto's (optioneel)
    accessoires/     <- minimaal 20 foto's (optioneel)
```

Foto's mogen zijn:
- Jouw eigen kledingfoto's
- Productfoto's van Zara / H&M / Zalando
- Google Images resultaten
- Mix van gedragen kleding en losstaande producten

**Meer foto's = beter resultaat.** Streef naar minimaal 20 per categorie, liever 50+.

## Na het trainen

Het getrainde model wordt opgeslagen als `kleding_model_trained/`. Daarna pas je in de
classifier notebook (kleding_classifier_v3) alleen de modelnaam aan:
```python
MODEL_NAAM = 'kleding_model_trained'  # was: 'wargoninnovation/wargon-clothing-classifier'
```

## Stap 1 — Installeren & importeren

In [ ]:
# !pip install transformers torch torchvision pillow matplotlib scikit-learn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np
import json
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Imports geslaagd!')
print(f'   Apparaat: {DEVICE}')
if DEVICE == 'cpu':
    print(f'   Let op: trainen op CPU is langzamer dan op GPU (~10-30 min voor kleine dataset)')

## Stap 2 — Trainingsdata inladen

Controleer of je mappenstructuur klopt en hoeveel foto's per categorie je hebt.

In [ ]:
TRAININGSMAP = 'kleding_training'
EXTENSIES    = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
MIN_FOTOS    = 10  # minimaal aantal foto's per categorie

if not os.path.isdir(TRAININGSMAP):
    print(f'Map "{TRAININGSMAP}" niet gevonden!')
    print(f'Maak de map aan met submappen per categorie (zie intro).')
else:
    categorieen = sorted([d for d in Path(TRAININGSMAP).iterdir() if d.is_dir()])

    if not categorieen:
        print(f'Geen submappen gevonden in "{TRAININGSMAP}"')
    else:
        print(f'Gevonden categorieen:')
        alle_paden = []
        alle_labels = []
        label_naar_id = {cat.name: i for i, cat in enumerate(categorieen)}
        id_naar_label = {i: cat.name for i, cat in enumerate(categorieen)}
        problemen = []

        for cat in categorieen:
            fotos = [f for f in cat.iterdir() if f.suffix.lower() in EXTENSIES]
            status = 'OK' if len(fotos) >= MIN_FOTOS else f'TE WEINIG (min. {MIN_FOTOS})'
            print(f'   {cat.name:20s}: {len(fotos):>3} fotos  [{status}]')
            if len(fotos) < MIN_FOTOS:
                problemen.append(cat.name)
            for f in fotos:
                alle_paden.append(f)
                alle_labels.append(label_naar_id[cat.name])

        print(f'\nTotaal: {len(alle_paden)} fotos, {len(categorieen)} categorieen')

        if problemen:
            print(f'\nWaarschuwing: te weinig fotos in: {", ".join(problemen)}')
            print(f'Voeg meer fotos toe voor betere resultaten.')
        else:
            print(f'\nData ziet er goed uit! Klaar voor training.')

## Stap 3 — Dataset klaarmaken

We splitsen de data in trainings- (80%) en validatieset (20%), en voegen data augmentation toe.

In [ ]:
class KledingDataset(Dataset):
    def __init__(self, paden, labels, transform):
        self.paden    = paden
        self.labels   = labels
        self.transform = transform

    def __len__(self):
        return len(self.paden)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paden[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color=(200, 200, 200))
        return self.transform(img), self.labels[idx]

# Transformaties — trainingset krijgt augmentation voor meer variatie
trein_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Train/validatie split (80/20)
trein_paden, val_paden, trein_labels, val_labels = train_test_split(
    alle_paden, alle_labels,
    test_size=0.2,
    random_state=42,
    stratify=alle_labels  # zorgt voor gelijke verdeling per categorie
)

trein_dataset = KledingDataset(trein_paden, trein_labels, trein_transform)
val_dataset   = KledingDataset(val_paden,   val_labels,   val_transform)

trein_loader = DataLoader(trein_dataset, batch_size=16, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=0)

print(f'Dataset klaar!')
print(f'   Trainingset:  {len(trein_dataset)} fotos')
print(f'   Validatieset: {len(val_dataset)} fotos')
print(f'   Batch grootte: 16')

## Stap 4 — Model laden en aanpassen

We laden het bestaande kledingmodel en vervangen alleen de laatste laag zodat het jouw categorieen leert.

In [ ]:
BASE_MODEL = 'wargoninnovation/wargon-clothing-classifier'

print('Model laden...')
processor = AutoImageProcessor.from_pretrained(BASE_MODEL)
model     = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(categorieen),
    ignore_mismatched_sizes=True  # vervangt de laatste laag
)

# Bevries alle lagen behalve de laatste — sneller trainen, minder overfitting
for naam, param in model.named_parameters():
    if 'classifier' not in naam:
        param.requires_grad = False

model = model.to(DEVICE)

trainbare = sum(p.numel() for p in model.parameters() if p.requires_grad)
totaal    = sum(p.numel() for p in model.parameters())
print(f'Model klaar!')
print(f'   {len(categorieen)} uitvoerklassen: {", ".join(id_naar_label[i] for i in range(len(categorieen)))}')
print(f'   Trainbare parameters: {trainbare:,} van {totaal:,} totaal')

## Stap 5 — Trainen

Het model wordt getraind. Op CPU duurt dit 10-30 minuten afhankelijk van je dataset.
Je ziet per epoch de voortgang en nauwkeurigheid.

In [ ]:
# Trainingsparameters
EPOCHS    = 10
LR        = 2e-4   # leersnelheid
GEDULD    = 3      # stop als validatie niet verbetert na X epochs (early stopping)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

beste_val_acc   = 0.0
geduld_teller   = 0
geschiedenis    = {'trein_verlies': [], 'trein_acc': [], 'val_verlies': [], 'val_acc': []}

print(f'Training gestart! ({EPOCHS} epochs, LR={LR})')
print('-' * 60)

for epoch in range(EPOCHS):
    # --- Trainingsfase ---
    model.train()
    trein_verlies = 0.0
    trein_correct = 0

    for imgs, labels in trein_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        uitvoer = model(pixel_values=imgs).logits
        verlies = criterion(uitvoer, labels)
        verlies.backward()
        optimizer.step()
        trein_verlies += verlies.item()
        trein_correct += (uitvoer.argmax(1) == labels).sum().item()

    # --- Validatiefase ---
    model.eval()
    val_verlies   = 0.0
    val_correct   = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            uitvoer = model(pixel_values=imgs).logits
            verlies = criterion(uitvoer, labels)
            val_verlies   += verlies.item()
            val_correct   += (uitvoer.argmax(1) == labels).sum().item()

    trein_acc = trein_correct / len(trein_dataset)
    val_acc   = val_correct   / len(val_dataset)
    trein_l   = trein_verlies / len(trein_loader)
    val_l     = val_verlies   / len(val_loader)

    geschiedenis['trein_verlies'].append(trein_l)
    geschiedenis['trein_acc'].append(trein_acc)
    geschiedenis['val_verlies'].append(val_l)
    geschiedenis['val_acc'].append(val_acc)

    scheduler.step()

    print(f'Epoch {epoch+1:>2}/{EPOCHS}  '
          f'trein: verlies={trein_l:.3f} acc={trein_acc*100:.1f}%  '
          f'val: verlies={val_l:.3f} acc={val_acc*100:.1f}%',
          end='')

    # Sla het beste model op
    if val_acc > beste_val_acc:
        beste_val_acc = val_acc
        geduld_teller = 0
        model.save_pretrained('kleding_model_trained')
        processor.save_pretrained('kleding_model_trained')
        # Sla ook de label-mapping op
        with open('kleding_model_trained/label_mapping.json', 'w') as f:
            json.dump(id_naar_label, f, ensure_ascii=False, indent=2)
        print('  <- beste model opgeslagen!')
    else:
        geduld_teller += 1
        print(f'  (geduld: {geduld_teller}/{GEDULD})')

    if geduld_teller >= GEDULD:
        print(f'\nEarly stopping na epoch {epoch+1} (geen verbetering in {GEDULD} epochs)')
        break

print(f'\nTraining klaar! Beste validatie-nauwkeurigheid: {beste_val_acc*100:.1f}%')

## Stap 6 — Resultaten bekijken

In [ ]:
# Grafiek van verlies en nauwkeurigheid per epoch
epochs_gedaan = len(geschiedenis['trein_acc'])
x = range(1, epochs_gedaan + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x, geschiedenis['trein_verlies'], label='Training', color='#e8a87c')
ax1.plot(x, geschiedenis['val_verlies'],   label='Validatie', color='#6b8cba')
ax1.set_title('Verlies per epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Verlies')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(x, [a*100 for a in geschiedenis['trein_acc']], label='Training', color='#e8a87c')
ax2.plot(x, [a*100 for a in geschiedenis['val_acc']],   label='Validatie', color='#6b8cba')
ax2.set_title('Nauwkeurigheid per epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Nauwkeurigheid (%)')
ax2.set_ylim(0, 105)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Trainingsresultaten — beste val. nauwkeurigheid: {beste_val_acc*100:.1f}%',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Beste model opgeslagen in: kleding_model_trained/')

## Stap 7 — Gedetailleerd rapport per categorie

In [ ]:
# Laad het beste opgeslagen model
model_best = AutoModelForImageClassification.from_pretrained('kleding_model_trained')
model_best.eval().to(DEVICE)

# Voorspellingen op de validatieset
alle_voorspeld = []
alle_werkelijk = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        uitvoer = model_best(pixel_values=imgs).logits
        alle_voorspeld.extend(uitvoer.argmax(1).cpu().tolist())
        alle_werkelijk.extend(labels.tolist())

label_namen = [id_naar_label[i] for i in range(len(categorieen))]
print('Rapport per categorie:')
print('=' * 50)
print(classification_report(
    alle_werkelijk,
    alle_voorspeld,
    target_names=label_namen,
    zero_division=0
))

## Stap 8 — Model gebruiken in de classifier

Na het trainen pas je in **kleding_classifier_v3.ipynb** alleen Stap 2 aan:

```python
# Oud:
MODEL_NAAM = 'wargoninnovation/wargon-clothing-classifier'

# Nieuw (jouw getrainde model):
MODEL_NAAM = 'kleding_model_trained'
```

En Stap 3 (LABEL_NAMEN) pas je aan zodat die overeenkomt met jouw categoriemappen:

```python
# Laad automatisch de labels van jouw getrainde model
import json
with open('kleding_model_trained/label_mapping.json') as f:
    id_naar_label = {int(k): v for k, v in json.load(f).items()}
```